# Phase 10 · Mise en exploitation : carnet de travail

Carte d'identité du modèle, reproductibilité sur l'échantillon, versionnage, liste opérationnelle et intégration
CRM (valeurs par défaut validées). Le jeu de test n'est pas lu.

In [1]:
# --- Environment: the project package and the recorded results --------------------------------
import json
import sys
from pathlib import Path

RACINE = Path.cwd()
for candidat in [RACINE, *RACINE.parents]:
    if (candidat / "src" / "churn_saas").is_dir():
        RACINE = candidat
        sys.path.insert(0, str(candidat / "src"))
        break

import pandas as pd

from churn_saas.notebook import afficher_tableau

## 1. Registre et modèle en service

In [2]:
registre = json.loads((RACINE / "resultats" / "registre_modeles.json").read_text(encoding="utf-8"))
afficher_tableau(pd.DataFrame(registre).drop(columns=["fichier_sha256", "empreinte_entrainement", "empreinte_gold"]),
                 titre="Registre des modèles")
aliases = json.loads((RACINE / "resultats" / "aliases_modeles.json").read_text(encoding="utf-8"))
print("Champion :", aliases["champion"]["nom"], aliases["champion"]["version"],
      "| précédent :", (aliases.get("precedent") or {}).get("version", "—"))

nom,version,date_entrainement,fichier,famille,copies,commit
churn_saas_servi,1.1,2026-10-03,churn_saas_servi_v1.1_20261003.joblib,regression_logistique,1,fede8a8


Champion : churn_saas_servi 1.1 | précédent : —


## 2. Mêmes scores sur l'échantillon (reconstruction depuis le code et les données)

In [3]:
import importlib.util

from churn_saas import config
from churn_saas.donnees import typer_pour_modele
from churn_saas.features import executer_pipeline, parties_du_decoupage
from churn_saas.industrialisation.scoring import preparer
from churn_saas.modelisation import construire_baseline

specification = importlib.util.spec_from_file_location("servi", RACINE / "tools" / "modele_servi.py")
servi = importlib.util.module_from_spec(specification)
specification.loader.exec_module(servi)
parties = parties_du_decoupage(executer_pipeline(config.FICHIER_COMPLET, config.FICHIER_CATALOGUE))
X, y = parties.X_entrainement, parties.y_entrainement.astype(int)
modele = servi.construire_servi(construire_baseline, "sigmoid", config)(X).fit(X, y)
brut = pd.read_csv(config.FICHIER_ECHANTILLON, dtype=str, encoding="utf-8-sig")
catalogue = pd.read_csv(config.FICHIER_CATALOGUE, dtype=str, encoding="utf-8-sig")
entree = typer_pour_modele(preparer(brut, catalogue=catalogue).drop(columns=["churn"], errors="ignore"))
reference = pd.read_csv(RACINE / "resultats" / "scores_echantillon_reference.csv")
ecart = (pd.Series(modele.predict_proba(entree)[:, 1]) - reference["proba"]).abs().max()
print(f"Écart maximal avec les scores de référence : {ecart:.1e} (tolérance 1e-9)")

Écart maximal avec les scores de référence : 5.0e-13 (tolérance 1e-9)


## 3. Liste opérationnelle (échantillon), avec groupe témoin

In [4]:
from churn_saas.industrialisation.liste import construire_liste

liste = construire_liste(modele, entree, brut["client_id"].set_axis(entree.index),
                         pd.to_numeric(brut["valeur_vie_client_eur"], errors="coerce").set_axis(entree.index),
                         entree["revenu_mensuel_recurrent_eur"], X, "churn_saas_servi 1.1",
                         capacite=5, genere_le="2026-10-01", graine=202610)
afficher_tableau(liste.head(8), titre="Les premières lignes de la liste")
print(liste["Action recommandée"].value_counts().to_string())

Priorité,Compte,Action recommandée,Risque de départ,Risque (%),Valeur client estimée (k€),Revenu mensuel (€),Gain attendu d'un contact (€),Pourquoi ce compte ?,Déjà contacté,Généré le,Modèle
1,CLI-000847,À contacter ce mois,Très élevé,92,173.3,6605.0,39706.0,Dernière connexion il y a 30 jours (augmente le risque) ; client depuis 1 mois (augmente le risque) ; aucune intégration active (augmente le risque),Nouveau,2026-10-01,churn_saas_servi 1.1
2,CLI-004120,À contacter ce mois,Modéré,32,233.8,5380.0,18589.0,6 ticket(s) de support en 3 mois (augmente le risque) ; 4 intégration(s) active(s) (réduit le risque) ; 2 retard(s) de paiement en 12 mois (augmente le risque),Nouveau,2026-10-01,churn_saas_servi 1.1
3,CLI-000228,À contacter ce mois,Faible,7,784.6,63786.0,13519.0,306 utilisateurs actifs (réduit le risque) ; formule enterprise (réduit le risque) ; revenu mensuel de 63 786 € (augmente le risque),Nouveau,2026-10-01,churn_saas_servi 1.1
4,CLI-003394,Groupe témoin (pas de contact),Élevé,43,124.7,4753.0,13346.0,3 retard(s) de paiement en 12 mois (augmente le risque) ; client depuis 1 mois (augmente le risque) ; aucune intégration active (augmente le risque),Nouveau,2026-10-01,churn_saas_servi 1.1
5,CLI-002189,À contacter ce mois,Très élevé,92,50.9,4438.0,11608.0,Dernière connexion il y a 49 jours (augmente le risque) ; aucune intégration active (augmente le risque) ; 0 % des licences utilisées (augmente le risque),Nouveau,2026-10-01,churn_saas_servi 1.1
6,CLI-000283,À contacter ce mois,Modéré,30,136.4,6932.0,10117.0,6 ticket(s) de support en 3 mois (augmente le risque) ; satisfaction 2 sur 5 (augmente le risque) ; 6 % des licences utilisées (augmente le risque),Nouveau,2026-10-01,churn_saas_servi 1.1
7,CLI-004519,"En réserve (rentable, hors capacité)",Élevé,47,71.8,7294.0,8360.0,5 retard(s) de paiement en 12 mois (augmente le risque) ; client depuis 1 mois (augmente le risque) ; satisfaction 5 sur 5 (réduit le risque),Nouveau,2026-10-01,churn_saas_servi 1.1
8,CLI-000782,"En réserve (rentable, hors capacité)",Élevé,50,56.8,8657.0,6959.0,Formule enterprise (réduit le risque) ; 5 ticket(s) de support en 3 mois (augmente le risque) ; client depuis 6 mois (augmente le risque),Nouveau,2026-10-01,churn_saas_servi 1.1


Action recommandée
En réserve (rentable, hors capacité)    26
Pas d'action                            18
À contacter ce mois                      5
Groupe témoin (pas de contact)           1


## Journal de bord
- Aucun modèle sans carte d'identité ; scores reproduits à l'identique ; retour arrière en une commande.
- Liste en libellés métier ; carence de 2 mois ; **10 % de groupe témoin** pour mesurer l'efficacité réelle.
- À confirmer avec le commanditaire : CRM, identifiants, retours (`docs/INTEGRATION_CRM.md`).